In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [2]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [3]:
benchmarks = [
    # Low-sharing Apps
    "fdtd2d",
    "jacobi1d",
    "fastwalshtransform",
    "relu",
    "fft",
    "fir",
    "sssp",
    "convolution2d",
    "color",
    "bitonicsort",
    "pagerank",
    "simpleconvolution",

    # High-sharing Apps
    'gups',    
    'syr2k',
    'spmv',
    'mis', 
    'stencil2d',
    'gemm',
    'lu',
    'matrixmultiplication',
    'mm2',
    'mm3',
    'gaussian',
    'matrixtranspose',
    'im2col',
    'vgg16',
    'bfs',
]

In [4]:
num_chiplets = 36
ns = 0.000000001

In [5]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223939"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223939


In [6]:
chip_cols = [str(i) for i in range(36)]

In [15]:
"""
전체 캐시 라인에서 read-only shared / read-write shared 비율
"""

print("benchmark", end='\t')
print("read_only", end='\t')
print("read_write", end='\t')
print("non_shared")

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"No {trace_file}, skip")
        continue

    agg_dict = {col: 'sum' for col in chip_cols}
    agg_dict['rw'] = 'max'  # rw는 1이 한 번이라도 있으면 1(max)이 됨

    addr_summary = df.groupby('addr', as_index=True).agg(agg_dict)

    # 2. 전체 캐시 라인 수는 GroupBy 결과의 행 개수와 동일합니다.
    total_cachelines = len(addr_summary)

    # 3. 마스크(조건) 생성
    # - 공유 데이터 여부: 0보다 큰(접근된) 칩렛 개수가 1초과 (> 1)
    is_shared = (addr_summary[chip_cols] > 0).sum(axis=1) > 1

    # - Read-Write / Read-Only 여부
    is_rw = addr_summary['rw'] == 1
    is_ro = ~is_rw

    # 4. 최종 캐시 라인 개수 도출 (True인 것들의 개수를 셈)
    non_shared_cachelines = ((addr_summary[chip_cols] > 0).sum(axis=1) <= 1).sum()
    shared_read_only_cachelines = (is_shared & is_ro).sum()
    shared_read_write_cachelines = (is_shared & is_rw).sum()

    # (선택 사항) 만약 이후 분석을 위해 DataFrame 자체가 계속 필요하다면 아래처럼 추출
    shared_read_only_df = addr_summary[is_shared & is_ro].reset_index()
    shared_read_write_df = addr_summary[is_shared & is_rw].reset_index()

    shared_read_only_ratio = shared_read_only_cachelines / total_cachelines
    shared_read_write_ratio = shared_read_write_cachelines / total_cachelines
    non_shared_ratio = non_shared_cachelines / total_cachelines

    print(f"{benchmark}", end='\t')
    print(f"{shared_read_only_ratio}", end='\t')
    print(f"{shared_read_write_ratio}", end='\t')
    print(f"{non_shared_ratio}")

benchmark	read_only	read_write	non_shared
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fdtd2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/jacobi1d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fastwalshtransform_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/relu_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fft_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fir_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/sssp_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/convolution2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/color_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/t

In [97]:
"""
전체 캐시 라인에서:
- Non-Shared 비율 (단일 비율)
- Read-Write Shared 비율 (단일 비율)
- Read-Only Shared 비율 (칩렛 공유 개수에 따라 4개 구간으로 세분화)
"""

import os
import pandas as pd

# 구간 설정: (1, 9]는 2~9개, (9, 18]은 10~18개를 의미합니다.
bins = [1, 9, 18, 27, 36]
labels = ['2~9', '10~18', '19~27', '28~36']

# 헤더 출력
print("benchmark\tnon_shared\trw_shared\t" + "\t".join([f"ro_{lbl}" for lbl in labels]))

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"{benchmark}\tNo {trace_file}, skip")
        continue

    # 1. 파일 읽기
    df = pd.read_csv(trace_file, skipinitialspace=True)

    # 2. GroupBy 연산
    agg_dict = {col: 'sum' for col in chip_cols}
    agg_dict['rw'] = 'max'  

    addr_summary = df.groupby('addr', as_index=True).agg(agg_dict)
    total_cachelines = len(addr_summary)

    # 3. 주소별 공유 칩렛 개수(0보다 큰 컬럼 수) 및 RW 여부 마스킹
    shared_counts = (addr_summary[chip_cols] > 0).sum(axis=1)
    is_ro = addr_summary['rw'] == 0  # Read-Only: rw 최댓값이 0
    is_rw = ~is_ro

    is_shared = shared_counts > 1

    # 4. Non-Shared 및 전체 Read-Write Shared 비율 계산 (단일 값)
    non_shared_ratio = (shared_counts <= 1).sum() / total_cachelines
    rw_shared_ratio = (is_shared & is_rw).sum() / total_cachelines

    # 5. Read-Only Shared 구간별 세부 비율 계산
    # Read-Only이면서 공유된(2개 이상) 주소들만 추출
    ro_shared_counts = shared_counts[is_shared & is_ro]
    
    # 추출된 개수를 구간(Bin)별로 나누어 카운트
    # pd.cut이 각 주소의 칩렛 개수가 어느 구간에 속하는지 라벨링해줌
    ro_bins = pd.cut(ro_shared_counts, bins=bins, labels=labels, right=True)
    
    # 구간별 개수를 전체 캐시 라인 수로 나누어 비율(Ratio) 계산
    # value_counts()는 지정된 라벨(labels)에 맞춰 빈도를 세어줍니다.
    ro_bin_ratios = ro_bins.value_counts(sort=False) / total_cachelines

    # 6. 출력 포맷팅
    ro_ratios_str = "\t".join([f"{ro_bin_ratios.get(lbl, 0.0):.6f}" for lbl in labels])

    print(f"{benchmark}\t{non_shared_ratio:.6f}\t{rw_shared_ratio:.6f}\t{ro_ratios_str}")


benchmark	non_shared	rw_shared	ro_2~9	ro_10~18	ro_19~27	ro_28~36
fdtd2d	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fdtd2d_mesh_6x6_memory_trace.csv, skip
jacobi1d	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/jacobi1d_mesh_6x6_memory_trace.csv, skip
fastwalshtransform	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fastwalshtransform_mesh_6x6_memory_trace.csv, skip
relu	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/relu_mesh_6x6_memory_trace.csv, skip
fft	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fft_mesh_6x6_memory_trace.csv, skip
fir	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fir_mesh_6x6_memory_trace.csv, skip
sssp	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/sssp_mesh_6x6_memory_trace.csv, skip
convolution2d	No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/convolution2d_mesh_6x6_memory_trace.csv, skip
color	No /home/ehdgus7279/mgvm/results/memory-traces/test

In [ ]:
"""
전체 캐시 라인에서:
- True Sharing
- False Sharing
- Non-sharing
"""

print("benchmark", end='\t')
print("True", end='\t')
print("False", end='\t')
print("non_shared")

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"No {trace_file}, skip")
        continue

    df = df[df['shared_chiplets'] > 0].copy()

    # 1. 0~35번 칩렛의 데이터를 Numpy Array로 추출 (속도 최적화)
    # ※ 만약 컬럼명이 정수(0, 1..)라면 [i for i in range(36)] 으로 변경하세요.
    chiplet_cols = [str(i) for i in range(36)] 
    chiplet_data = df[chiplet_cols].values

    # 2. owner가 자신의 데이터에 접근했는지 확인
    # owner 컬럼의 값(0~35)을 인덱스로 활용하여, 각 행마다 owner 칩렛의 접근 횟수를 O(N)으로 빠르게 추출합니다.
    row_idx = np.arange(len(df))
    owner_idx = df['owner'].astype(int).values

    # 각 행별로 owner 칩렛의 접근 횟수를 가져옵니다.
    owner_access_counts = chiplet_data[row_idx, owner_idx]
    owner_accessed = owner_access_counts > 0

    # 3. 사용자 정의 로직에 따른 조건 마스크 생성
    # True Sharing: 접근한 칩렛이 2개 이상 (owner 포함 복수 이거나, owner 제외 복수)
    cond_true_sharing = (df['shared_chiplets'] >= 2)

    # Non-sharing: 접근한 칩렛이 1개이고, 그 1개가 owner인 경우
    cond_non_sharing = (df['shared_chiplets'] == 1) & owner_accessed

    # False Sharing: 접근한 칩렛이 1개이고, 그 1개가 owner가 아닌 다른 칩렛인 경우
    cond_false_sharing = (df['shared_chiplets'] == 1) & ~owner_accessed

    # 4. 비율 계산
    total = len(df)
    true_ratio = cond_true_sharing.sum() / total * 100
    non_ratio = cond_non_sharing.sum() / total * 100
    false_ratio = cond_false_sharing.sum() / total * 100

    # (선택) 데이터프레임에 sharing_type 컬럼 추가
    # df['sharing_type'] = np.select(
    #     [cond_true_sharing, cond_non_sharing, cond_false_sharing],
    #     ['True Sharing', 'Non-sharing', 'False Sharing'],
    #     default='Unknown'
    # )

    print(f"{benchmark}", end='\t')
    print(f"{true_ratio}", end='\t')
    print(f"{false_ratio}", end='\t')
    print(f"{non_ratio}")

benchmark	True	False	non_shared
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fdtd2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/jacobi1d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fastwalshtransform_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/relu_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fft_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/fir_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/sssp_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/convolution2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expanded/color_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/test/expand

In [25]:
def pc_metrics_interval(df):
    """
    True/False Sharing이 발생한 Read-Only 주소에 대해 
    시간대별, PC별 캐시 라인 수와 총 접근 횟수를 집계합니다.
    """

    time_interval=10000

    # ---------------------------------------------------------
    # 1. True Sharing + False Sharing 여부 판별
    # ---------------------------------------------------------

    chiplet_cols = [str(i) for i in range(36)]
    chiplet_data = df[chiplet_cols].values

    row_idx = np.arange(len(df))
    owner_idx = df['owner'].astype(int).values

    # 해당 행의 주소에 owner 칩렛이 접근했는지 여부 (Numpy 기반 초고속 연산)
    owner_accessed = chiplet_data[row_idx, owner_idx] > 0

    # True Sharing: 2개 이상 칩렛이 접근
    cond_true_sharing = df['shared_chiplets'] >= 2
    # False Sharing: 1개 칩렛만 접근했는데, 그게 owner가 아님
    cond_false_sharing = (df['shared_chiplets'] == 1) & ~owner_accessed

    # Shared 조건: 둘 중 하나라도 만족
    cond_shared = cond_true_sharing | cond_false_sharing

    # ---------------------------------------------------------
    # 2. Read-Only 주소 판별
    # ---------------------------------------------------------
    # ※ 주의: 데이터에 '명령어 종류(R/W)'를 나타내는 컬럼이 있다고 가정합니다.
    # 만약 데이터 자체가 이미 Read-Only만 모아둔 것이라면 이 과정은 생략 가능합니다.
    # 전체 트레이스에서 한 번이라도 Write('W')가 발생한 주소 목록 추출
    write_addrs = df[df['rw'] == 'W']['addr'].unique()
    # Write가 한 번도 발생하지 않은 주소만 True
    cond_read_only = ~df['addr'].isin(write_addrs)

    # ---------------------------------------------------------
    # 3. 데이터 필터링 전, 원본 df에 시간 정규화 및 interval 우선 적용
    # ---------------------------------------------------------
    # 기준점(t_min)을 원본 데이터 전체 기준으로 잡아야 정렬과 매핑이 어긋나지 않습니다.
    t_min = df['time'].min()
    # ns 변수는 기존처럼 정의되어 있다고 가정합니다.
    df['norm_time'] = (df['time'] - t_min) / ns
    df['interval'] = (df['norm_time'] // time_interval) * time_interval

    # 필터링
    filtered_df = df[cond_shared & cond_read_only].copy()

    # 전체 고유 캐시 라인의 개수 (원본 기준)
    total_cachelines = df['addr'].nunique()

    # ---------------------------------------------------------
    # 4. 시간 구간(Epoch) 설정 및 집계(Aggregation)
    # ---------------------------------------------------------

    # [4-1] filtered_df에서 PC별 Shared & Read-Only 접근 집계
    result_df = filtered_df.groupby(['interval', 'pc']).agg(
        cacheline_count=('addr', 'nunique'),     
        pc_shared_access_count=('access_count', 'sum') # 이름 변경: 해당 PC의 Shared RO 접근 횟수
    ).reset_index()

    # 전체 캐시라인 대비 비율
    result_df['shared_ro_cacheline_ratio'] = result_df['cacheline_count'] / total_cachelines


    # ---------------------------------------------------------
    # 5. 원본 df를 활용한 "전체 접근 횟수" 대비 비율 계산
    # ---------------------------------------------------------

    # [5-1] 해당 interval에서 발생한 '모든 PC'의 전체 접근 횟수 총합
    interval_total_df = df.groupby('interval').agg(
        interval_total_access=('access_count', 'sum')
    ).reset_index()

    # [5-2] 해당 interval에서 '해당 PC'가 발생시킨 전체 접근 횟수 총합 (선택 사항)
    # "PC 자신이 생성해낸 전체 주소에 대한 접근 횟수 대비"라는 말씀이 이 부분일 수 있어 추가했습니다.
    pc_interval_total_df = df.groupby(['interval', 'pc']).agg(
        pc_total_access=('access_count', 'sum')
    ).reset_index()

    # 데이터 병합 (Merge)
    result_df = pd.merge(result_df, interval_total_df, on='interval', how='left')
    result_df = pd.merge(result_df, pc_interval_total_df, on=['interval', 'pc'], how='left')

    # [5-3] 최종 비율 계산
    # 지표 A: (해당 구간 특정 PC의 Shared RO 접근 횟수) / (해당 구간의 전체 메모리 접근 횟수)
    result_df['shared_ro_access_ratio'] = result_df['pc_shared_access_count'] / result_df['interval_total_access']

    # 지표 B: (해당 구간 특정 PC의 Shared RO 접근 횟수) / (해당 구간 해당 PC가 발생시킨 전체 메모리 접근 횟수)
    # result_df['ratio_vs_pc_total'] = result_df['pc_shared_access_count'] / result_df['pc_total_access']


    # ---------------------------------------------------------
    # 6. 결과 정렬
    # ---------------------------------------------------------
    # 시간순으로 먼저 정렬 후, 동일 시간대에서는 Shared RO 접근 횟수가 많은 PC 순으로 정렬
    result_df = result_df.sort_values(by=['interval', 'pc_shared_access_count'], ascending=[True, False])

    result_dir = f"{trace_dir}/pc-metrics-interval"

    if not os.path.isdir(f'{result_dir}'):
        os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)
    
    result_df.to_csv(f"{result_dir}/{benchmark}_pc_metrics_interval.csv", \
                    index=False)
    
    return result_df

In [ ]:
def pc_metrics_interval_to_png(result_df, result_dir, benchmark):
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    import pandas as pd
    import numpy as np

    # ---------------------------------------------------------
    # 차트 글로벌 스타일 설정 (엑셀 스타일 및 고해상도 설정)
    # ---------------------------------------------------------
    plt.rcParams['figure.dpi'] = 300           # 화면 출력 해상도
    plt.rcParams['savefig.dpi'] = 300          # 저장 해상도 (논문용)
    plt.rcParams['font.family'] = 'sans-serif' # 깔끔한 폰트
    plt.rcParams['axes.axisbelow'] = True      # 그리드를 데이터 뒤로 배치

    # ---------------------------------------------------------
    # 1. 데이터 피벗 (interval을 X축 인덱스로, PC를 각 라인/영역으로)
    # ---------------------------------------------------------
    # 누락된 구간(interval)에서 선이나 영역이 끊기지 않도록 NaN을 0으로 채웁니다.

    # 영역형 차트용 데이터 (Cacheline Ratio)
    df_cache = result_df.pivot(index='interval', columns='pc', values='shared_ro_cacheline_ratio').fillna(0)
    df_access = result_df.pivot(index='interval', columns='pc', values='shared_ro_access_ratio').fillna(0)

    # 엑셀과 비슷한 느낌을 내기 위해 색상 팔레트 추출 (PC 개수만큼)
    cmap = plt.get_cmap('tab20')
    colors = [cmap(i % 20) for i in range(len(df_cache.columns))]

    # =========================================================
    # 1. 영역형 차트 (Shared RO Cacheline Ratio - 엑셀 첫 번째 사진)
    # =========================================================
    fig1, ax1 = plt.subplots(figsize=(10, 5))

    # 누적 영역(Stacked Area) 차트 생성
    ax1.stackplot(df_cache.index, 
                    [df_cache[col] for col in df_cache.columns], 
                    labels=df_cache.columns, 
                    colors=colors, alpha=0.95)

    # 테두리 및 그리드 설정 (엑셀 스타일: 가로선만 렌더링)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_color('#cccccc')
    ax1.spines['bottom'].set_color('#cccccc')
    ax1.yaxis.grid(True, linestyle='-', color='#e0e0e0', linewidth=0.7)
    ax1.xaxis.grid(False)

    # 축 설정
    ax1.set_ylim(0, 1.0)
    ax1.margins(x=0) # X축 양끝 빈 공간 제거
    ax1.set_ylabel('Shared RO Cacheline Ratio', fontsize=11, fontweight='bold')
    ax1.set_xlabel('Interval (Time)', fontsize=11, fontweight='bold')

    # X축 눈금 포맷팅 (겹치지 않게 개수 제한 및 45도 회전)
    ax1.xaxis.set_major_locator(ticker.MaxNLocator(nbins=20))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.1)) # 0.1 단위 눈금

    # 범례 설정 (그래프 밖 우측 상단에 배치)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), title='PC', frameon=False, fontsize=9)

    if not os.path.isdir(f'{result_dir}'):
        os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)
        
    # 저장 및 출력
    fig1.tight_layout()
    fig1.savefig(f'{result_dir}/{benchmark}_cacheline_ratio_area.png', bbox_inches='tight')

    # =========================================================
    # 2. 실선형 차트 (Shared RO Access Ratio - 엑셀 두 번째 사진)
    # =========================================================
    fig2, ax2 = plt.subplots(figsize=(10, 5))

    # 각 PC별로 Line 차트 생성 (선 두께 2.5로 가독성 확보)
    for idx, col in enumerate(df_access.columns):
        ax2.plot(df_access.index, df_access[col], label=col, color=colors[idx], linewidth=2.5)

    # 테두리 및 그리드 설정
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.spines['left'].set_color('#cccccc')
    ax2.spines['bottom'].set_color('#cccccc')
    ax2.yaxis.grid(True, linestyle='-', color='#e0e0e0', linewidth=0.7)
    ax2.xaxis.grid(False)

    # 축 설정
    # 엑셀 사진처럼 데이터 최댓값에 맞춰 ylim을 설정 (보통 0~1 사이)
    ax2.set_ylim(0, max(1.0, df_access.max().max() * 1.1)) 
    ax2.margins(x=0)
    ax2.set_ylabel('Shared RO Access Ratio', fontsize=11, fontweight='bold')
    ax2.set_xlabel('Interval (Time)', fontsize=11, fontweight='bold')

    # X축 눈금 포맷팅
    ax2.xaxis.set_major_locator(ticker.MaxNLocator(nbins=20))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(0.1))

    # 범례 설정
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1), title='PC', frameon=False, fontsize=9)

    # 저장 및 출력
    fig2.tight_layout()
    fig2.savefig(f'{result_dir}/{benchmark}_cacheline_access_ratio.png', bbox_inches='tight')
    plt.close()

In [26]:
for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"No {trace_file}, skip")
        continue

    df = pd.read_csv(trace_file, skipinitialspace=True)
    df = pc_metrics_interval(df)
    
    # result_dir = f"{trace_dir}/fig"
    # pc_metrics_interval_to_png(df, result_dir, benchmark)
    

No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/fdtd2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/jacobi1d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/fastwalshtransform_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/relu_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/fft_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/fir_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/sssp_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/convolution2d_mesh_6x6_memory_trace.csv, skip
No /home/ehdgus7279/mgvm/results/memory-traces/20260221-223939/expanded/color_mesh_6x6_memory_trace.c

In [19]:
result_dir = f"{trace_dir}/fig"
benchmark = "matrixmultiplication"
trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)
df = pc_metrics_interval(df)
df

In [ ]:
"""
벤치마크 별 PC 개수
"""
# 루프 밖에서 디렉토리 생성 (모든 벤치마크 공통)
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
os.makedirs(trace_dir, exist_ok=True, mode=0o777)

addr_num_results = []
pc_num_results = []

max_pc_num = 0

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    print(f"Processing {benchmark}: {trace_file}")
    
    # 파일 존재 체크
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    try:
        df = pd.read_csv(trace_file)
        df['cacheline'] = df['addr'].values & ~0x3F
        
        pc_stats = df.groupby('pc')['cacheline'].nunique().reset_index(name='unique_cnt')

        if max_pc_num < len(pc_stats):
            max_pc_num = len(pc_stats)
        
        columns = ['benchmark'] + [str(i) for i in range(max_pc_num)]
        row_data = {'benchmark': benchmark}
        
        # 0~99 컬럼 채우기 (PC 순서대로, 부족분 0)
        for i in range(max_pc_num):
            if i < len(pc_stats):
                row_data[str(i)] = pc_stats.iloc[i]['unique_cnt']
            else:
                row_data[str(i)] = 0
        
        row_df = pd.DataFrame([row_data], columns=columns)
        addr_num_results.append(row_df)
        pc_num_results.append(pd.DataFrame([{
            'benchmark': benchmark,
            'pc_num': len(pc_stats),
        }]))

        print(f"  OK: {len(pc_stats)} PCs processed")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

# 모든 결과 합치기
addr_num_df = pd.concat(addr_num_results, ignore_index=True)
addr_num_df = addr_num_df.fillna(0).astype({'benchmark': str})  # 타입 정리

pc_num_df = pd.concat(pc_num_results, ignore_index=True)
pc_num_df = pc_num_df.fillna(0).astype({'benchmark': str})

# 저장 (CSV 추천, 엑셀도 가능)
addr_num_file = f"{trace_dir}/pc-metrics/pc_to_addrs.csv"
pc_num_file = f"{trace_dir}/pc-metrics/benchmark_to_pc_num.csv"

if not os.path.isdir(f'{trace_dir}/pc-metrics'):
    os.makedirs(f'{trace_dir}/pc-metrics', exist_ok=True, mode=0o777)

addr_num_df.to_csv(addr_num_file, index=False)
pc_num_df.to_csv(pc_num_file, index=False)
print(f"\nSaved: {addr_num_file} (shape: {addr_num_df.shape})")
print(f"\nSaved: {pc_num_file} (shape: {pc_num_df.shape})")


Processing gemm: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv


In [ ]:
"""
벤치마크 별 Shared PC의 Access count ratio 및 addr 개수
"""

# 루프 밖에서 디렉토리 생성 (모든 벤치마크 공통)
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/high-share'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    print(f"Processing {benchmark}: {trace_file}")
    
    # 파일 존재 체크
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    df = pd.read_csv(
        trace_file,
        skipinitialspace=True,
    )

    # 2. 시간 정규화 및 사이클 변환
    t_min = df['time'].min()
    df['time'] = (df['time'] - t_min) / ns
    max_time = df['time'].max()
    term_cycle = 10000
    
    grouped = df.groupby(['pc', 'addr'])
    grouped = grouped.reset_index()

    high_share = grouped[grouped['chiplets'] >= 2]

    high_pc_metrics = high_share.groupby('pc').agg({
        'chiplets': 'mean',     # 평균 공유도
        'count': 'sum',         # 총 count (또는 'mean', 'count')
        'addr': 'nunique'
    }).reset_index()

    high_pc_metrics = high_pc_metrics.rename(columns={
        'chiplets': 'high_chiplets',
        'count': 'high_count',
        'addr': 'high_share_addrs'
    })

    high_pc_metrics['high_count_ratio'] = high_pc_metrics['high_count'] /grouped['count'].sum()
    high_pc_metrics.to_csv(f"{result_dir}/{benchmark}_high_share.csv")
    

Processing gemm: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv
Processing lu: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/lu_mesh_6x6_memory_trace.csv
Processing spmv: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/spmv_mesh_6x6_memory_trace.csv
Processing mm3: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm3_mesh_6x6_memory_trace.csv
Processing mm2: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm2_mesh_6x6_memory_trace.csv
Processing matrixmultiplication: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/matrixmultiplication_mesh_6x6_memory_trace.csv
Processing syr2k: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/syr2k_mesh_6x6_memory_trace.csv
Processing gups: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gups_mesh_6x6_memory_trace.csv
Processing mis: /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mis_mesh_6x6_memory_trace.csv
Proc

In [ ]:
"""
벤치마크 별 top_n addr에 대한 PC 개수, address count ratio 등
"""

# 루프 밖에서 디렉토리 생성 (모든 벤치마크 공통)
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/addr-metrics'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

n = 1000

columns = [
    'start',
    'end',
    'benchmark',
    'pc_num',
    'chiplet_avg',
    f'{n}_vs_shared',
    f'{n}_vs_total',
]

for benchmark in benchmarks:
    all_results = []
    
    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    print(f"Processing {benchmark}: {trace_file}")
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    # 데이터 로드 & 전처리 (기존 코드 동일)
    df = pd.read_csv(
        trace_file,
        skipinitialspace=True,
        converters={"metrics": parse_access_str}  # 읽으면서 바로 딕셔너리로 변환
    )

    access_dicts = df['access'].apply(parse_access_str)
    chips_df = pd.DataFrame(access_dicts.tolist()).reindex(columns=range(36)).fillna(0).astype(int)
    df = pd.concat([df[['time', 'addr', 'pc']], chips_df], axis=1)
    
    # groupby로 pc/addr 메트릭스 계산
    chip_cols = list(range(36))
    agg_rules = {col: 'sum' for col in chip_cols}

    # 시간 정규화
    t_min = df['time'].min()
    df['time'] = (df['time'] - t_min) / ns
    max_time = df['time'].max()

    term_cycles = 10000
    division = math.ceil(max_time / term_cycles)

    for i in range(division):
        period_start = i*term_cycles
        period_end = (i+1)*term_cycles
        period_df = df[(df['time']>=period_start) & (df['time']<period_end)]

        grouped = period_df.groupby(['pc', 'addr']).agg({col: 'sum' for col in chip_cols})
        grouped['chiplets'] = (grouped[chip_cols] != 0).sum(axis=1)
        grouped['count'] = grouped[chip_cols].sum(axis=1)
        grouped = grouped.reset_index()

        sorted_df = grouped.sort_values('count', ascending=False).reset_index(drop=True)
        sorted_df = sorted_df[sorted_df['chiplets']>1]
        total_count = grouped['count'].sum()
        total_shared_count = sorted_df['count'].sum()
        
        # data
        row_data = {'benchmark': benchmark}

        top_count = sorted_df.iloc[:n]['count'].sum()
        pc_num = sorted_df.iloc[:n]['pc'].nunique()
        chiplet_avg = sorted_df.iloc[:n]['chiplets'].mean()

        row_data['start'] = period_start
        row_data['end'] = period_end
        row_data['pc_num'] = pc_num
        row_data['chiplet_avg'] = chiplet_avg
        row_data[f'{n}_vs_shared'] = top_count / total_shared_count
        row_data[f'{n}_vs_total'] = top_count / total_count

        all_results.append(pd.DataFrame([row_data], columns=columns))

    final_df = pd.concat(all_results, ignore_index=True)
    mean_row = final_df.mean(numeric_only=True)
    final_df.loc[len(final_df)] = mean_row
    final_df.to_csv(f"{result_dir}/{benchmark}_{n}_addrs_metrics.csv", index=False)

In [ ]:
"""
벤치마크 별, 시간마다의 PC metric (access count, shared chiplets, access count ratio, number of cachelines)
"""

# 루프 밖에서 디렉토리 생성 (모든 벤치마크 공통)
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/shared-pc-metrics'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

for benchmark in benchmarks:
    # if benchmark != 'gemm':
    #     break
    
    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    print(f"Processing {benchmark} at {trace_file}")
    
    # 1. 데이터 로드 + 칩렛 확장 (한 번만!)
    df = pd.read_csv(trace_file, skipinitialspace=True)
    df = pd.concat([df[['time', 'addr', 'pc']], chips_df], axis=1)
    
    # 2. 시간 정규화
    t_min = df['time'].min()
    df['time'] = (df['time'] - t_min) / ns
    max_time = df['time'].max()
    
    term_cycles = 10000
    division = math.ceil(max_time / term_cycles)
    
    # 3. ★ 루프 없이 시간 구간화 (pd.cut으로 벡터화!)
    df['period'] = pd.cut(df['time'], bins=division, right=False, labels=range(division))
    
    chip_cols = list(range(36))
    agg_rules = {col: 'sum' for col in chip_cols}
    
    # 4. ★ 한 번에 모든 구간 집계 (period 추가!)
    grouped = (df.groupby(['period', 'pc', 'addr'], observed=True)
               .agg(agg_rules)
               .reset_index())
    
    grouped['chiplets'] = (grouped[chip_cols] != 0).sum(axis=1)
    grouped['count'] = grouped[chip_cols].sum(axis=1)
    
    # 5. 구간별 shared PC metrics
    shared = grouped[grouped['chiplets'] >= 2]
    
    shared_pc_metrics = (shared.groupby(['period', 'pc'])
                        .agg({
                            'chiplets': 'mean',
                            'count': 'sum',
                            'addr': 'nunique',
                        })
                        .reset_index()
                        .rename(columns={
                            'chiplets': 'mean_shared_chiplets',
                            'count': 'total_shared_access_count',
                            'addr': 'num_cachelines'
                        }))

    # 6. 구간별 비율 계산
    period_totals = grouped.groupby('period')['count'].sum().reset_index(name='period_total')
    
    # [추가됨] 6-1. 구간(Period)별 총 Active PC 개수와 총 Unique Address 개수 계산
    # df는 필터링되지 않은 전체 데이터이므로 여기서 구해야 전체 통계가 나옵니다.
    period_stats = df.groupby('period', observed=True).agg({
        'pc': 'nunique',
        'addr': 'nunique'
    }).reset_index().rename(columns={
        'pc': 'total_active_pcs_in_period',   # 해당 구간에 활동한 총 PC 수
        'addr': 'total_unique_addrs_in_period' # 해당 구간에 접근된 총 주소 수
    })

    # 기존 period_totals와 병합 (Total Count + Unique Counts)
    period_context = period_totals.merge(period_stats, on='period', how='left')

    # shared_pc_metrics와 병합
    merged_metrics = shared_pc_metrics.merge(period_context, on='period', how='left')

    merged_metrics['ratio'] = (merged_metrics['total_shared_access_count'] / 
                            merged_metrics['period_total']).fillna(0)

    # 7. ★ 각 period 내 PC별 addr count의 평균/표준편차
    pc_stats = (grouped.groupby(['period', 'pc'])['count']
           .agg(['mean', 'std'])  # addr별 count의 평균/표준편차
           .reset_index()
           .rename(columns={'mean': 'addr_count_mean', 'std': 'addr_count_std'}))

    # 8. 기존 metrics와 병합
    final_metrics = merged_metrics.merge(pc_stats, on=['period', 'pc'], how='left')
    final_metrics['addr_count_mean'] = final_metrics['addr_count_mean'].fillna(0)
    final_metrics['addr_count_std'] = final_metrics['addr_count_std'].fillna(0)

    final_metrics.to_csv(f"{result_dir}/{benchmark}_shared_pc_metrics.csv", index=False)

Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv
Processing pagerank at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/pagerank_mesh_6x6_memory_trace.csv
Processing gemm at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv
Processing lu at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/lu_mesh_6x6_memory_trace.csv
Processing spmv at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/spmv_mesh_6x6_memory_trace.csv
Processing mm3 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm3_mesh_6x6_memory_trace.csv
Processing mm2 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm2_mesh_6x6_memory_trace.csv
Processing matrixmultiplication at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/matrixmultiplication_mesh_6x6_memory_trace.csv
Processing syr2k at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/syr2k_mesh

In [16]:
import os
import pandas as pd
import numpy as np

# 경로 설정
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/pc-metrics'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

# 모든 벤치마크의 결과를 모을 리스트
all_benchmarks_data = []

for benchmark in benchmarks:
    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    print(f"Processing {benchmark} at {trace_file}")
    
    # 1. 데이터 로드 및 파싱
    df = pd.read_csv(trace_file, skipinitialspace=True)
    
    # access string 파싱 (parse_access_str 함수가 정의되어 있다고 가정)
    access_dicts = df['access'].apply(parse_access_str)
    
    # 칩렛 접근 정보 데이터프레임 생성
    chips_df = pd.DataFrame(access_dicts.tolist()).reindex(columns=range(36)).fillna(0).astype(int)
    
    # [수정 1] df_merged 만들 때 'addr' 컬럼도 포함해야 함
    # 기존: df_merged = pd.concat([df[['pc']], chips_df], axis=1)
    df_merged = pd.concat([df[['pc', 'addr']], chips_df], axis=1)
    
    # 2. PC 정규화 (필요시 사용)
    t_min = df_merged['pc'].min()
    df_merged['pc'] = df_merged['pc'] - t_min
    
    chip_cols = list(range(36))
    
    # [수정 2] 집계 규칙(agg_rules)에 addr의 고유 개수(nunique) 계산 추가
    agg_rules = {col: 'sum' for col in chip_cols}
    agg_rules['addr'] = 'nunique'  # ★ 핵심: PC별로 addr가 몇 종류인지 셈
    
    # 3. PC 별 집계
    grouped = df_merged.groupby(['pc'], observed=True).agg(agg_rules).reset_index()
    
    # 컬럼명 정리 ('addr' -> 'num_cachelines'로 변경하여 혼동 방지)
    grouped.rename(columns={'addr': 'num_cachelines'}, inplace=True)
    
    # 4. 각 PC별 메트릭 계산
    grouped['shared_chiplets'] = (grouped[chip_cols] != 0).sum(axis=1)
    grouped['access_count'] = grouped[chip_cols].sum(axis=1)
    
    # 5. 벤치마크 전체 Total 계산
    total_benchmark_access = grouped['access_count'].sum()
    
    # [수정 3] 전체 유니크 주소 개수는 원본 df에서 구해야 정확함 
    # (grouped['num_cachelines'].sum()은 중복 주소가 PC간에 겹칠 경우 과대계상 될 수 있으나, 
    #  Instruction 별로 접근 주소 영역이 다르다면 비슷할 수 있음. 하지만 정확히는 원본 사용 권장)
    total_benchmark_cachelines = df['addr'].nunique()
    
    # 6. Shared PC 필터링
    shared_pc_df = grouped[grouped['shared_chiplets'] >= 2].copy()
    
    # 7. 결과 데이터 정리
    shared_pc_df['benchmark'] = benchmark
    shared_pc_df['ratio'] = shared_pc_df['access_count'] / total_benchmark_access
    
    # [수정 4] 이미 위에서 계산된 'num_cachelines' 컬럼을 그대로 사용
    shared_pc_df['num_cachelines_ratio'] = shared_pc_df['num_cachelines'] / total_benchmark_cachelines
    
    # 저장할 컬럼 선택
    result_cols = ['benchmark', 'pc', 'access_count', 'ratio', 'shared_chiplets', 'num_cachelines', 'num_cachelines_ratio']
    all_benchmarks_data.append(shared_pc_df[result_cols])

# 8. 전체 결과 병합 및 저장 (루프 밖에서 수행)
if all_benchmarks_data:
    final_result_df = pd.concat(all_benchmarks_data, ignore_index=True)
    
    # 보기 좋게 정렬 (벤치마크별, Count 높은 순)
    final_result_df = final_result_df.sort_values(by=['benchmark', 'access_count'], ascending=[True, False])
    
    output_path = f"{result_dir}/all_benchmarks_shared_pc_metrics.csv"
    final_result_df.to_csv(output_path, index=False)
    print(f"Saved merged metrics to {output_path}")
else:
    print("No data collected.")

Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv
Processing pagerank at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/pagerank_mesh_6x6_memory_trace.csv
Processing gemm at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv
Processing lu at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/lu_mesh_6x6_memory_trace.csv
Processing spmv at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/spmv_mesh_6x6_memory_trace.csv
Processing mm3 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm3_mesh_6x6_memory_trace.csv
Processing mm2 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm2_mesh_6x6_memory_trace.csv
Processing matrixmultiplication at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/matrixmultiplication_mesh_6x6_memory_trace.csv
Processing syr2k at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/syr2k_mesh

In [ ]:
import os
import pandas as pd
import numpy as np

"""
시간 별 addr의 access count
"""

# 경로 설정
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/access-count-per-time'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

# 모든 벤치마크의 결과를 모을 리스트
all_benchmarks_data = []

for benchmark in benchmarks:
    # if benchmark != 'bfs':
        # break

    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    print(f"Processing {benchmark} at {trace_file}")
    
    # 1. 데이터 로드 및 파싱
    df = pd.read_csv(trace_file, skipinitialspace=True)

    # 2. 시간 정규화
    t_min = df['time'].min()
    df['time'] = (df['time'] - t_min) / ns
    max_time = df['time'].max()

    # access string 파싱 (parse_access_str 함수가 정의되어 있다고 가정)
    access_dicts = df['access'].apply(parse_access_str)

    # 칩렛 접근 정보 데이터프레임 생성
    chips_df = pd.DataFrame(access_dicts.tolist()).reindex(columns=range(36)).fillna(0).astype(int)

    # [수정 1] df_merged 만들 때 'addr' 컬럼도 포함해야 함
    # 기존: df_merged = pd.concat([df[['pc']], chips_df], axis=1)
    df_merged = pd.concat([df[['pc', 'addr']], chips_df], axis=1)

    term_cycles = 10000
    division = math.ceil(max_time / term_cycles)

    # 3. ★ 루프 없이 시간 구간화 (pd.cut으로 벡터화!)
    df_merged['period'] = pd.cut(df['time'], bins=division, right=False, labels=range(division))

    chip_cols = list(range(36))
    agg_rules = {col: 'sum' for col in chip_cols}

    # 3. PC 별 집계
    grouped = df_merged.groupby(['period', 'addr'], observed=True).agg(agg_rules).reset_index()
    grouped['access_count'] = grouped[chip_cols].sum(axis=1)
    grouped['shared_chiplets'] = (grouped[chip_cols] != 0).sum(axis=1)
    grouped = grouped.drop(chip_cols, axis=1)

    repeated_addrs = grouped.groupby('addr')['period'].nunique().reset_index()
    repeated_addrs = repeated_addrs[repeated_addrs['period']>1]

    grouped = grouped[grouped['addr'].isin(repeated_addrs['addr'])]
    
    grouped = grouped.groupby('addr').agg({
        'access_count': 'mean',
        'shared_chiplets': 'mean',
        'period': 'nunique',
    }).reset_index()
    
    grouped.to_csv(f"{result_dir}/{benchmark}_access_count_per_addr_time.csv", index=False)


Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv
Processing pagerank at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/pagerank_mesh_6x6_memory_trace.csv
Processing gemm at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv
Processing lu at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/lu_mesh_6x6_memory_trace.csv
Processing spmv at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/spmv_mesh_6x6_memory_trace.csv
Processing mm3 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm3_mesh_6x6_memory_trace.csv
Processing mm2 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm2_mesh_6x6_memory_trace.csv
Processing matrixmultiplication at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/matrixmultiplication_mesh_6x6_memory_trace.csv
Processing syr2k at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/syr2k_mesh

In [92]:
import os
import pandas as pd
import numpy as np

# 경로 설정
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

# 모든 벤치마크의 결과를 모을 리스트
results = []

for benchmark in benchmarks:
    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    print(f"Processing {benchmark} at {trace_file}")
    
    # 1. 데이터 로드
    df = pd.read_csv(trace_file, skipinitialspace=True)
    
    # 2. Stride 계산 (Cache Line 단위)
    # diff() 후 바로 abs() 적용
    df['stride'] = df['addr'].diff().abs()
    
    # 0인 stride (동일 주소 연속 접근) 제거 
    # (비율 계산 시 분모에서 제외할지 포함할지는 의도에 따라 결정. 여기선 제외로 구현)
    df = df[df['stride'] != 0].copy()
    
    # Cache Line (64B) 단위로 변환
    df['stride'] = df['stride'] / 64
    
    # 전체 샘플 수 (분모)
    total_count = len(df)
    
    if total_count == 0:
        print(f"  Warning: No valid strides for {benchmark}")
        continue

    # 3. 구간별 비율 계산 (Boolean Indexing 후 개수 세기)
    # 각 조건에 맞는 행의 개수(sum)를 전체 개수로 나눕니다.
    res = {
        'benchmark': benchmark,
        '1':   (df['stride'] == 1).sum() / total_count,
        '2':   (df['stride'] == 2).sum() / total_count,
        '~4':  ((df['stride'] > 2)  & (df['stride'] <= 4)).sum() / total_count,
        '~8':  ((df['stride'] > 4)  & (df['stride'] <= 8)).sum() / total_count,
        '~16': ((df['stride'] > 8)  & (df['stride'] <= 16)).sum() / total_count,
        '~32': ((df['stride'] > 16) & (df['stride'] <= 32)).sum() / total_count,
        '>32': (df['stride'] > 32).sum() / total_count,
    }
    
    results.append(res)

# 결과 저장
results_df = pd.DataFrame(results)

# 컬럼 순서 강제 지정 (딕셔너리는 순서가 보장되지 않을 수 있음)
columns = ['benchmark', '1', '2', '~4', '~8', '~16', '~32', '>32']
results_df = results_df[columns]

output_path = f"{result_dir}/strides_per_benchmark.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv
Processing pagerank at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/pagerank_mesh_6x6_memory_trace.csv
Processing gemm at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/gemm_mesh_6x6_memory_trace.csv
Processing lu at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/lu_mesh_6x6_memory_trace.csv
Processing spmv at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/spmv_mesh_6x6_memory_trace.csv
Processing mm3 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm3_mesh_6x6_memory_trace.csv
Processing mm2 at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/mm2_mesh_6x6_memory_trace.csv
Processing matrixmultiplication at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/matrixmultiplication_mesh_6x6_memory_trace.csv
Processing syr2k at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/syr2k_mesh

In [19]:
benchmark = 'bfs'

trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"

print(f"Processing {benchmark} at {trace_file}")

# 1. 데이터 로드 및 파싱
df = pd.read_csv(trace_file, skipinitialspace=True)

# 2. 시간 정규화
t_min = df['time'].min()
df['time'] = (df['time'] - t_min) / ns
max_time = df['time'].max()

# access string 파싱 (parse_access_str 함수가 정의되어 있다고 가정)
access_dicts = df['access'].apply(parse_access_str)

# 칩렛 접근 정보 데이터프레임 생성
chips_df = pd.DataFrame(access_dicts.tolist()).reindex(columns=range(36)).fillna(0).astype(int)

# [수정 1] df_merged 만들 때 'addr' 컬럼도 포함해야 함
# 기존: df_merged = pd.concat([df[['pc']], chips_df], axis=1)
df_merged = pd.concat([df[['pc', 'addr']], chips_df], axis=1)

term_cycles = 10000
division = math.ceil(max_time / term_cycles)

# 3. ★ 루프 없이 시간 구간화 (pd.cut으로 벡터화!)
df_merged['period'] = pd.cut(df['time'], bins=division, right=False, labels=range(division))

chip_cols = list(range(36))
agg_rules = {col: 'sum' for col in chip_cols}

# 3. PC 별 집계
grouped = df_merged.groupby(['period', 'addr'], observed=True).agg(agg_rules).reset_index()
grouped['access_count'] = grouped[chip_cols].sum(axis=1)
grouped['shared_chiplets'] = (grouped[chip_cols] != 0).sum(axis=1)
grouped = grouped.drop(chip_cols, axis=1)

repeated_addrs = grouped.groupby('addr')['period'].nunique().reset_index()
repeated_addrs = repeated_addrs[repeated_addrs['period']>1]

grouped = grouped[grouped['addr'].isin(repeated_addrs['addr'])]

grouped = grouped.groupby('addr').agg({
    'access_count': 'mean',
    'shared_chiplets': 'mean',
    'period': 'nunique',
}).reset_index()


Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv


In [79]:
benchmark = 'bfs'

trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"

print(f"Processing {benchmark} at {trace_file}")

# 1. 데이터 로드 및 파싱
df = pd.read_csv(trace_file, skipinitialspace=True)
df = df.sort_values(by='time')

Processing bfs at /home/ehdgus7279/mgvm/results/memory-traces/20260211-235812/bfs_mesh_6x6_memory_trace.csv


In [80]:
df['stride'] = df['addr'].diff().abs()
df['stride'] /= 64

In [ ]:
stride_counts = df['stride'].value_counts().reset_index()
stride_counts = stride_counts.sort_values(by='stride').reset_index()
stride_counts = stride_counts[stride_counts['stride']!=0]
stride_counts['ratio'] = stride_counts['count'] / stride_counts['count'].sum()
stride_counts

,index,stride,count,ratio
1,0,1.0,45852,0.104912
2,1,2.0,35168,0.080466
3,2768,3.0,12,0.000027
4,4,4.0,5465,0.012504
5,10887,5.0,4,0.000009
...,...,...,...,...
86256,33611,16625572.0,1,0.000002
86257,33616,16625606.0,1,0.000002
86258,33617,16625608.0,1,0.000002
86259,33610,16625822.0,1,0.000002


In [35]:

import os
import pandas as pd
import numpy as np

# 경로 설정
trace_dir = f'{username}/mgvm/results/memory-traces/{date}'
result_dir = f'{trace_dir}/pc-metrics'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

# 모든 벤치마크의 결과를 모을 리스트
all_benchmarks_data = []

for benchmark in benchmarks:
    trace_file = f"{base_dir}/{date}/{benchmark}_mesh_6x6_memory_trace.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    # 1. 데이터 로드 및 파싱
    df = pd.read_csv(trace_file, skipinitialspace=True)
    
    # 1. 주소(addr)별로 접근한 고유 PC(pc)의 개수를 셉니다.
    addr_counts = df.groupby('addr')['pc'].nunique()

    # 2. 2개 이상의 PC에서 접근된 주소만 필터링합니다.
    shared_addrs = addr_counts[addr_counts > 1]

    # 결과 확인
    print(f"Benchmark: {benchmark}")
    print(f"shared_addrs {len(shared_addrs)}, {df['addr'].nunique()}: {len(shared_addrs)/df['addr'].nunique()}")

Benchmark: bfs
shared_addrs 65541, 135001: 0.48548529270153556


KeyboardInterrupt: 

In [27]:
import os
import pandas as pd
import numpy as np

# 경로 설정
trace_dir = f'{username}/mgvm/results/memory-traces/{date}/shared-pc-metrics'
result_dir = f'{trace_dir}'

if not os.path.isdir(f'{result_dir}'):
    os.makedirs(f'{result_dir}', exist_ok=True, mode=0o777)

for benchmark in benchmarks:
    trace_file = f"{trace_dir}/{benchmark}_shared_pc_metrics.csv"
    
    if not os.path.exists(trace_file):
        print(f"  Skip: No file {trace_file}")
        continue
    
    # 1. 데이터 로드 및 파싱
    df = pd.read_csv(trace_file, skipinitialspace=True)
    
    df['cacheline_ratio'] = df['num_cachelines'] / df['total_unique_addrs_in_period']

    df.to_csv(f"{trace_file}", index=False)